# SI_robustness_gpt_validation.ipynb

**Purpose:** Use an LLM to independently judge whether each candidate name-match
in `1_sample_2.csv` is a genuine self-promotion match, then use those judgments
as "ground truth" to compute precision/recall/F1 for every similarity-threshold
pair -> **Fig. S8 input**.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Changes from the original notebook** (`1_gpt_matching_test.ipynb`):
- **Removed the hardcoded API key** that was committed in plaintext in the
  original notebook. If that key is still active, treat it as compromised and
  rotate/revoke it immediately -- it was exposed in a file you were about to
  make public. This notebook reads the key from an environment variable
  instead.
- The manual GPT-disagreement overrides (Section 3 below) are preserved from
  the original, but are tied to the **specific** 100-row sample drawn by
  `SI_robustness_gender_ratio.ipynb` with its fixed random seed. If
  you ever redraw that sample with a different seed, these row indices will
  no longer point at the same records and this section needs to be redone by
  re-reviewing the new GPT/algorithm disagreements.

**Required input file:** `2_result/1_sample_2.csv` (from
`SI_robustness_gender_ratio.ipynb`)

**Required environment variable:** `GWDG_API_KEY` (or whichever provider you
use -- set this before launching Jupyter, e.g. `export GWDG_API_KEY=...`)

**Outputs:**
- `2_result/1_sample_2_testresult.csv` -- Fig. S8 input (`SI_robustness_S8_S9.R` writes it to `tables/fig_s8.csv`)
- `2_result/1_sample_2_gpt_disagreements.csv` -- the review table displayed in Section 3 (contains names; keep private)



In [ ]:
import os
import time

import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RESULT_DIR = "2_result"

### 1. API configuration

Read from an environment variable -- **never hardcode API keys in a notebook
that will be committed to a public repository.**

Set this before launching Jupyter, e.g. on macOS/Linux:
```bash
export GWDG_API_KEY=your_key_here
jupyter notebook
```


In [ ]:
api_key = os.environ.get("GWDG_API_KEY")
if api_key is None:
    raise EnvironmentError(
        "Set the GWDG_API_KEY environment variable before running "
        "(e.g. `export GWDG_API_KEY=...`)."
    )

client = OpenAI(api_key=api_key, base_url="https://chat-ai.academiccloud.de/v1")
MODEL = "openai-gpt-oss-120b"


def check_name_similarity(first, last, display, max_retries=5):
    """Ask the LLM whether `display` (a Twitter display name or handle)
    plausibly refers to the same person as `first`/`last`. Returns "Yes",
    "No", or None if the request ultimately failed after retries."""
    prompt = f"""
    You are given a person\'s first name, last name, and display name.

    First name: {first}
    Last name: {last}
    Display name: {display}

    Determine whether the display name refers to the same person (i.e., if
    it contains, abbreviates, or combines the first and last name).
    Answer with one word: "Yes" or "No".
    """

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):  # rate limit
                wait_time = 2 ** attempt + np.random.random()
                print(f"Rate limit hit. Waiting {wait_time:.1f} seconds...")
                time.sleep(wait_time)
            else:
                print(f"Error: {e}")
                return None
    return None

### 2. Load the validation sample and get GPT judgments

In [ ]:
matching_sample = pd.read_csv(f"{RESULT_DIR}/1_sample_2.csv")

results = []
for _, row in matching_sample.iterrows():
    result = check_name_similarity(row["first_name"], row["last_name"], row["most_likely_name"])
    results.append(result)
    time.sleep(0.5)  # be polite to the rate limit

matching_sample["gpt_match_test"] = results
matching_sample["gpt_match_test"] = matching_sample["gpt_match_test"].map({"Yes": 1, "No": 0})

### 3. Manual review of GPT/algorithm disagreements

After reviewing the cases where the GPT judgment and the algorithm's
max_score-based decision disagreed, the following rows were manually confirmed
(by inspecting the actual names/handles) to be genuine matches that GPT had
mis-labeled. This is a one-time human-in-the-loop correction to the ground
truth, not something to regenerate automatically.

> **Important:** these row labels correspond to the specific 100-row sample
> drawn with `random_state=2026` in step 1. If you redraw the sample, re-run
> the disagreement check below and manually re-review before reusing this
> list.

In [ ]:
print("Rows where GPT and the algorithm disagreed (review before trusting ground truth):")
disagreements = matching_sample[matching_sample["gpt_match_test"] != matching_sample["self_pro"]][
    ["most_likely_name", "last_name", "first_name", "gpt_match_test", "self_pro"]
]
# Saved with the other intermediate files (NOT in tables/): it contains real
# author names and Twitter display names, so it must not be published.
disagreements.to_csv(f"{RESULT_DIR}/1_sample_2_gpt_disagreements.csv")
disagreements

In [ ]:
manual_overrides = [72, 77, 80, 81, 82, 89, 95, 99]  # confirmed genuine matches on manual review
for idx in manual_overrides:
    if idx in matching_sample.index:
        matching_sample.loc[idx, "gpt_match_test"] = 1
    else:
        print(f"Warning: index {idx} not present in this sample -- skipping override.")

### 4. Threshold sweep against the GPT + manual ground truth -> Fig. S8

In [ ]:
def threshold_test(scored_sample, general_threshold, chinese_threshold):
    """Recompute self_pro for the validation sample at a candidate pair of
    similarity thresholds (same logic as in step 1\'s threshold_test, applied
    here to the small labeled validation sample instead of the full data)."""
    x = scored_sample.copy()
    x["self_pro"] = (x["max_score"] > general_threshold).astype(int)
    x["self_pro"] = x.apply(
        lambda row: 0 if (row["max_score"] < chinese_threshold) and (row["chinese_name"] == 1)
        else row["self_pro"],
        axis=1,
    )
    return x[[
        "author_id", "gpt_match_test", "max_score", "last_name", "first_name",
        "Index_firstname", "first_name_revised", "most_likely_name",
        "chinese_name", "self_pro",
    ]]


def evaluate_model(df, true_col="gpt_match_test", pred_col="self_pro"):
    y_true, y_pred = df[true_col], df[pred_col]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
threshold_result = []
for general_t in np.arange(0.5, 1.01, 0.1):
    for chinese_t in np.arange(0.5, 1.01, 0.1):
        test = threshold_test(matching_sample, round(general_t, 2), round(chinese_t, 2))
        metrics = evaluate_model(test)
        threshold_result.append({
            "general_threshold": round(general_t, 2),
            "chinese_threshold": round(chinese_t, 2),
            **metrics,
        })

threshold_result = pd.DataFrame(threshold_result)
threshold_result.to_csv(f"{RESULT_DIR}/1_sample_2_testresult.csv", index=False)
print(f"Wrote {RESULT_DIR}/1_sample_2_testresult.csv ({len(threshold_result)} threshold pairs)")

best_row = threshold_result.loc[threshold_result["f1"].idxmax()]
best_f1 = best_row["f1"]
best_thresholds = best_row[["general_threshold", "chinese_threshold"]].to_dict()
print(f"Best F1: {best_f1:.3f} at {best_thresholds}")